## Import

Set the project root and GPU visibility before importing `torch` or `vllm`. This notebook is meant for interactive checking; production runs should still use `main.py` with `configs/BRIDGE.yaml`.


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "main.py").exists():
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

# Respect an externally configured CUDA_VISIBLE_DEVICES; otherwise use this notebook default.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1,2,3,4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "False")

list_gpu = [gpu.strip() for gpu in os.environ["CUDA_VISIBLE_DEVICES"].split(",") if gpu.strip()]
num_gpus = len(list_gpu)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("num_gpus:", num_gpus)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
import random
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [ ]:
import torch
import transformers
from vllm import SamplingParams

In [ ]:
from dataset.process import format_chat
from dataset.task import get_cls_for_all_tasks, get_type_task
from util.runtime import parse_prompt_mode

In [ ]:
def seed_everything(seed):
    seed = int(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    transformers.set_seed(seed)
    print(f"seed everything: {seed}")


seed = "42"
seed_everything(seed=seed)

## Args

Configure the model, prompt mode, and task here. Environment variables can override the notebook defaults without editing the file:

- `BENCHMARK_MODEL_NAME`
- `BENCHMARK_PROMPT_MODE`
- `BENCHMARK_TASK_NAME`
- `MODEL_FILE`
- `NUM_SAMPLE`

Supported prompt modes are `direct`, `cot`, `direct-N-shot`, and `cot-N-shot`.


In [ ]:
model_name = "Qwen3-30B-A3B-Instruct-2507"
print("model_name:", model_name)

In [ ]:
prompt_mode = "direct"
prompt_base, num_example = parse_prompt_mode(prompt_mode)
print("prompt_mode:", prompt_mode)
print("prompt_base:", prompt_base)
print("num_example:", num_example)

In [ ]:
class EmptyArgs:
    def __init__(self):
        self.gpus = list_gpu
        self.engine = "vllm"
        self.model_name = model_name
        self.model_file = "configs/dict_model_path.json"
        self.gpu_memory_util = "0.8"
        self.max_token_all = 10 * 1024
        self.max_token_input = 8 * 1024
        self.max_token_output = self.max_token_all - self.max_token_input
        self.dir_dataset = ["dataset_raw"]
        self.dir_result = ["result"]
        self.dir_result_save = self.dir_result[0]


args = EmptyArgs()
if not args.gpus:
    raise ValueError("No GPU id found. Set CUDA_VISIBLE_DEVICES before loading the model.")
if not Path(args.model_file).exists():
    raise FileNotFoundError(f"Model mapping file does not exist: {args.model_file}")

for k, v in vars(args).items():
    print(f"{k}: {v}")

## Model and Tokenizer Initialization

This cell loads the model into GPU memory. Run it only after confirming the GPU list, model name, token budget, and model mapping file above.


In [ ]:
from model.init import init_model

In [ ]:
# Initialize the model and tokenizer.
tokenizer, model = init_model(args)
model_name = args.model_name
print("runtime model_name:", model_name)
print("max_token_all:", args.max_token_all)
print("max_token_input:", args.max_token_input)
print("max_token_output:", args.max_token_output)

In [ ]:
sampling_params = SamplingParams(
    seed=seed,
    temperature=0.0,
    max_tokens=args.max_token_output,
)
print(sampling_params)

## Task class

The task data must follow the standard benchmark format: each row needs `id`, `split`, `language`, `type`, `instruction`, `input`, and `output`. Few-shot examples are loaded from `example/` or `example_cot/` according to `prompt_mode`.


In [ ]:
DICT_TYPE_TASK_CLS, DICT_TASK_CLS = get_cls_for_all_tasks()
print(f"Registered tasks: {len(DICT_TASK_CLS)}")
list(DICT_TASK_CLS.keys())[:10]

In [ ]:
task_name = "ADE-Identification"
if task_name not in DICT_TASK_CLS:
    raise KeyError(f"Task not registered: {task_name}")

In [ ]:
task = DICT_TASK_CLS[task_name](args=args, task=task_name)
task.setup(tokenizer=tokenizer, prompt_mode=prompt_mode)
print("Task:", task.name)
print("Task type:", get_type_task(task))
print("test size:", len(task.dataset_test))
print("examples:", len(task.examples))

## Inference - test one

Use one sample to inspect prompt formatting and model output before running a larger batch.


In [ ]:
if not task.dataset_test:
    raise ValueError(f"Task has no test samples: {task_name}")
idx = min(5, len(task.dataset_test) - 1)
data = task.dataset_test[idx]
data

In [ ]:
formatted_input = format_chat(
    model_name=model_name,
    tokenizer=tokenizer,
    data=data,
    max_token_input=args.max_token_input,
    examples=task.examples,
    chat_kwargs=getattr(args, "chat_kwargs", {}),
)
print(formatted_input)

In [ ]:
output = model.generate(formatted_input, sampling_params=sampling_params, use_tqdm=False)
for output_one in output:
    generated_text = output_one.outputs[0].text
    print(f"Prompt:\n\t{formatted_input}")
    print("-" * 50)
    print(f"Generated text:\n\t{generated_text}")

## Inference - task

Run a bounded subset for interactive testing. Set `NUM_SAMPLE` to control the sample size without editing the notebook.


In [ ]:
num_sample = 100
if num_sample <= 0:
    raise ValueError("num_sample must be positive and the selected split must not be empty.")
print("num_sample:", num_sample)

In [ ]:
# Format the input text with the selected prompt mode.
list_input = []
list_num_token = []
for idx_data, dict_data in enumerate(task.dataset_test[:num_sample]):
    input_llm = format_chat(
        model_name=model_name,
        tokenizer=tokenizer,
        data=dict_data,
        max_token_input=args.max_token_input,
        examples=task.examples,
        chat_kwargs=getattr(args, "chat_kwargs", {}),
    )
    list_input.append(input_llm)
    len_token_input = len(tokenizer.encode(input_llm))
    list_num_token.append(len_token_input)
    if len_token_input > args.max_token_input:
        print(
            f"Input exceeds max token limit: idx={idx_data}, "
            f"tokens={len_token_input}, max={args.max_token_input}"
        )
print(f"Data size: {len(list_input)}")

In [ ]:
if len(list_input) > 1:
    print(list_input[1])
else:
    print(list_input[0])

In [ ]:
dict_stat_num_token = pd.Series(list_num_token).describe().to_dict()
print(dict_stat_num_token)

In [ ]:
# Count prompts that still exceed the configured input budget after formatting.
num_truncate = sum(num_token > args.max_token_input for num_token in list_num_token)
proportion_truncate = num_truncate / len(list_num_token) if list_num_token else 0.0
print(f"The number of data that may be truncated: {num_truncate}")
print(f"The proportion of data that may be truncated: {proportion_truncate:.2%}")

In [ ]:
plt.hist(list_num_token, bins=30, alpha=0.7)
plt.title('Token Count Distribution')
plt.xlabel('Number of Tokens')
plt.ylabel('Frequency')
plt.show()

In [ ]:
list_pred = []
output = model.generate(list_input, sampling_params=sampling_params, use_tqdm=True)
for output_one in output:
    generated_text = output_one.outputs[0].text
    list_pred.append(generated_text)

if len(list_pred) != len(list_input):
    raise ValueError(f"Prediction count mismatch: {len(list_pred)} vs {len(list_input)}")

## Result

Attach predictions to the sampled task rows, parse them with the task-specific parser, and compute quick metrics. This section is for smoke testing; production result files should be written by `main.py`.


In [ ]:
for input_text, pred_text in zip(list_input, list_pred):
    print(f"Input text:\n\t{input_text}")
    print("-"*50)
    print(f"Generated text:\n\t{pred_text}")
    print("="*100)

In [ ]:
for idx_data, dict_data in enumerate(task.dataset_test[:num_sample]):
    dict_data["pred"] = list_pred[idx_data]
print(f"Attached predictions to {num_sample} samples.")

In [ ]:
list_eval_data = task.dataset_test[:num_sample]
list_pred_extracted = task.get_pred(list_eval_data, prompt_mode=prompt_mode)
list_label_extracted = task.get_label(list_eval_data, prompt_mode=prompt_mode)
print(f"Extracted {len(list_pred_extracted)} predictions and {len(list_label_extracted)} labels.")

### Performance

In [ ]:
list_pred_extracted, num_failed = task.get_pred_none(
    list_pred=list_pred_extracted[:num_sample],
    list_label=list_label_extracted[:num_sample],
)
print(f"The number of failed data: {num_failed} ({num_failed / num_sample:.2%})")

In [ ]:
task_type = get_type_task(task)
if task_type == "classification":
    dict_performance = task.get_performance(list_pred_extracted, list_label_extracted)
else:
    dict_performance, _ = task.get_performance(list_pred_extracted, list_label_extracted)
print("task_type:", task_type)


In [ ]:
dict_performance

## End

Release GPU memory by restarting the kernel when you finish interactive testing.


In [ ]:
print('Done!')